# Using the DynamicFunction Class

## Introduction

DynamicFunction is an abstract function wrapper that has multiplexed bind and callback functionality. It inherits from DynamicCallable and specializes behavior for free functions. Under the hood, it uses MethodMultiplexer for both binding (__get__) and calling (__call__).

This tutorial highlights how to use DynamicFunction, with emphasis on its method multiplexer functionality for binding and callback.

### Table of Contents
- Importing
- Core behavior
- Binding and calling with multiplexers
- Descriptor usage
- Advanced: Switching call strategies
- Examples
- FAQs


## Importing

In [ ]:
from baseobjects.functions import DynamicFunction
from baseobjects.functions import MethodMultiplexer


## Core Behavior

- DynamicFunction works best with standalone functions.
- default_bind_method inherited from DynamicCallable is "bind_builtin".
- default_call_method is "call_wrapped".
- Properties bind_method and call_method map to MethodMultiplexer selections.


## Binding and Calling with Multiplexers

Both binding and calling go through MethodMultiplexer instances held by the DynamicFunction:
- bind_multiplexer handles descriptor binding (__get__).
- call_multiplexer handles invocation (__call__).

You can set which strategy to use at runtime via bind_method and call_method.


## Descriptor Usage

DynamicFunction instances can be placed on classes as descriptors. When accessed via an instance, binding occurs through the bind_multiplexer.

In [ ]:
class Tools:
    pass

# Function to wrap
def add(a, b):
    return a + b

# Attach a DynamicFunction to the class
Tools.add = DynamicFunction(add)

# Use from an instance
obj = Tools()
print('2+3 =', obj.add(2,3))

# Switch binding behavior if needed
Tools.add.bind_method = 'bind_builtin'
print('4+5 =', obj.add(4,5))


## Advanced: Switching Call Strategies

You can define additional call implementations on a subclass and switch between them using the call_multiplexer.

In [ ]:
class Calc(DynamicFunction):
    def call_wrapped(self, *args, **kwargs):
        return self._wrapped_(*args, **kwargs)
    
    def call_safe(self, *args, **kwargs):
        try:
            return self._wrapped_(*args, **kwargs)
        except Exception as e:
            return f"Error: {e}"

calc = Calc(add)
print('default:', calc(10, 2))

# Switch to safe mode
calc.call_method = 'call_safe'
print('safe:', calc(10, 0))

# Switch back
calc.call_method = 'call_wrapped'
print('wrapped:', calc(7, 8))


## Examples

### Example 1: Decorating a function-like API with dynamic strategies

In [ ]:
def mul(a, b):
    return a * b

class Op(DynamicFunction):
    def call_wrapped(self, *args, **kwargs):
        return self._wrapped_(*args, **kwargs)
    
    def call_logged(self, *args, **kwargs):
        result = self._wrapped_(*args, **kwargs)
        print(f"call_logged -> {args} -> {result}")
        return result

multiply = Op(mul)
print('mul:', multiply(3, 4))

multiply.call_method = 'call_logged'
print('mul logged:', multiply(5, 6))


## FAQs

Q: How does MethodMultiplexer relate to DynamicFunction?\n
A: DynamicFunction contains two MethodMultiplexer instances that choose the binding and call strategies. You can select strategies by assigning bind_method and call_method.

Q: Can DynamicFunction handle methods?\n
A: It can wrap callables, but DynamicMethod is the better fit for instance methods as it automatically passes the bound self when calling. Use DynamicMethod for method use cases.
